# Standalone tokenizer embedding adaptation experiment

This notebook does not import AlignTune. It continually extends a Hugging Face tokenizer, adapts a model's embeddings without shrinking padded vocabularies, compares `mean` and `mean_of_constituents` initialization, then runs short full and PEFT training experiments.

It installs `tokenizer-extension` from GitHub and runs independently in Google Colab without importing AlignTune.

In [ ]:
%pip install -q -U transformers datasets peft accelerate tokenizers git+https://github.com/taidopurason/tokenizer-extension.git


In [ ]:
# Run the installation cell once in a fresh Colab runtime.
from __future__ import annotations

import copy
import gc
from pathlib import Path

import torch
from datasets import Dataset
from peft import LoraConfig, PeftModel, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer

from tokenizer_extension.extension import extend_tokenizer
from tokenizer_extension.pruning import LeafFrequencyPruner
from tokenizer_extension.train_vocab_extension import train_vocab_extension

torch.manual_seed(7)
print('PyTorch:', torch.__version__)

In [ ]:
# Qwen has a padded embedding matrix, which is useful for testing grow-only behavior.
MODEL_NAME = 'Qwen/Qwen2.5-0.5B'
EXTENSION_SIZE = 32
MAX_LENGTH = 128
TRAIN_STEPS = 4
LEARNING_RATE = 5e-4

# The notebook initializes both methods, then trains this method in both modes.
INITIALIZATION_METHODS = ['mean', 'mean_of_constituents']
TRAINING_METHOD = 'mean_of_constituents'

# Corpus deliberately contains repeated Indic-script patterns for continual BPE training.
CORPUS = [
    'भारत में भाषा मॉडल के लिए बेहतर टोकनाइज़र प्रशिक्षण उपयोगी है।',
    'हिंदी और मराठी पाठ के लिए नई सबवर्ड इकाइयाँ सीखना महत्वपूर्ण है।',
    'मशीन लर्निंग मॉडल नई शब्दावली के साथ बेहतर प्रदर्शन कर सकता है।',
    'टोकनाइज़र विस्तार से लंबे शब्द छोटे टुकड़ों में कम टूटते हैं।',
    'बहुभाषी डेटा पर सतत प्रशिक्षण गुणवत्ता और कवरेज सुधारता है।',
    'नई भाषा के शब्दों के लिए एम्बेडिंग आरंभ करना आवश्यक है।',
] * 40

WORK_DIR = Path('artifacts/tokenizer_embedding_experiment').resolve()
WORK_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float32
print({'device': DEVICE, 'dtype': str(DTYPE), 'work_dir': str(WORK_DIR)})

In [ ]:
def vocab_summary(tokenizer, label):
    vocab = tokenizer.get_vocab()
    max_id = max(vocab.values()) if vocab else -1
    print({
        'label': label,
        'len(tokenizer)': len(tokenizer),
        'max_token_id': max_id,
        'required_rows': max_id + 1,
        'pad_token': tokenizer.pad_token,
        'eos_token': tokenizer.eos_token,
    })


def tokenizer_diff(base_tokenizer, target_tokenizer):
    base_vocab = base_tokenizer.get_vocab()
    target_vocab = target_tokenizer.get_vocab()

    retained = base_vocab.keys() & target_vocab.keys()
    new_tokens = target_vocab.keys() - base_vocab.keys()
    removed_tokens = base_vocab.keys() - target_vocab.keys()
    moved_tokens = {
        token: (base_vocab[token], target_vocab[token])
        for token in retained
        if base_vocab[token] != target_vocab[token]
    }

    return {
        'retained_tokens': len(retained),
        'new_tokens': {token: target_vocab[token] for token in sorted(new_tokens)},
        'removed_tokens': {token: base_vocab[token] for token in sorted(removed_tokens)},
        'moved_tokens': moved_tokens,
    }


base_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
vocab_summary(base_tokenizer, 'base tokenizer')

In [ ]:
# Continual BPE training creates candidate tokens; extend_tokenizer appends them to the base tokenizer.
extension = train_vocab_extension(
    tokenizer=base_tokenizer,
    corpus=CORPUS,
    extension_size=EXTENSION_SIZE,
    max_token_length=24,
)

target_tokenizer = extend_tokenizer(
    tokenizer=copy.deepcopy(base_tokenizer),
    new_vocab=extension['vocab'],
    new_merges=extension['merges'],
    n_tokens=EXTENSION_SIZE,
    alphabet='byte',
)

target_tokenizer.save_pretrained(WORK_DIR / 'extended_tokenizer')
vocab_summary(target_tokenizer, 'continually trained tokenizer')

diff = tokenizer_diff(base_tokenizer, target_tokenizer)
print('new token count:', len(diff['new_tokens']))
print('removed token count:', len(diff['removed_tokens']))
print('moved token count:', len(diff['moved_tokens']))
print('sample new tokens:', list(diff['new_tokens'].items())[:10])

## Local, padding-safe embedding adaptation

This intentionally does not use `tokenizer_extension.models.modify_embeddings()`. It preserves padded rows for extension, shrinks after pruning, copies retained rows by token string, and initializes only genuinely added tokens.

In [ ]:
def required_embedding_rows(tokenizer):
    vocab = tokenizer.get_vocab()
    return max(vocab.values()) + 1


def resolve_module_name(model, module):
    for name, candidate in model.named_modules():
        if candidate is module:
            return name
    raise ValueError(f'Could not resolve module name for {module.__class__.__name__}')


def random_like(row, initializer_range):
    values = torch.empty(row.shape, device=row.device, dtype=torch.float32)
    values.normal_(mean=0.0, std=initializer_range)
    return values.to(dtype=row.dtype)


def initialize_extended_token_embeddings(
    model,
    base_tokenizer,
    target_tokenizer,
    init_method,
):
    if init_method not in {'random', 'mean', 'mean_of_constituents'}:
        raise ValueError(f'Unsupported init method: {init_method}')

    old_vocab = base_tokenizer.get_vocab()
    new_vocab = target_tokenizer.get_vocab()
    retained_tokens = sorted(old_vocab.keys() & new_vocab.keys())
    new_tokens = sorted(new_vocab.keys() - old_vocab.keys())
    removed_tokens = sorted(old_vocab.keys() - new_vocab.keys())
    old_input = model.get_input_embeddings().weight.detach().clone()
    old_output_module = model.get_output_embeddings()
    tied = bool(getattr(model.config, 'tie_word_embeddings', False))
    old_output = None if tied or old_output_module is None else old_output_module.weight.detach().clone()
    old_output_bias = (
        None
        if tied or old_output_module is None or old_output_module.bias is None
        else old_output_module.bias.detach().clone()
    )

    current_rows = old_input.shape[0]
    final_vocab_size = len(old_vocab) + len(new_tokens) - len(removed_tokens)
    assert final_vocab_size == len(new_vocab)
    desired_rows = final_vocab_size if removed_tokens else max(current_rows, final_vocab_size)

    if desired_rows != current_rows:
        model.resize_token_embeddings(desired_rows)

    input_embeddings = model.get_input_embeddings()
    output_embeddings = model.get_output_embeddings()
    if input_embeddings.weight.shape[0] < final_vocab_size:
        raise RuntimeError('Embedding resize did not provide target tokenizer capacity')

    old_vocab_ids = sorted(set(old_vocab.values()))
    old_vocab_id_set = set(old_vocab_ids)
    initializer_range = float(getattr(model.config, 'initializer_range', 0.02))
    input_mean = old_input[old_vocab_ids].float().mean(dim=0).to(old_input.dtype)
    output_mean = None if old_output is None else old_output[old_vocab_ids].float().mean(dim=0).to(old_output.dtype)
    constituent_fallbacks = []

    def source_ids(token):
        ids = [piece.id for piece in base_tokenizer._tokenizer.model.tokenize(token)]
        ids = [idx for idx in ids if idx in old_vocab_id_set]
        if not ids:
            constituent_fallbacks.append(token)
        return ids

    with torch.no_grad():
        # Copy first from snapshots. This remains correct when pruning shifts IDs.
        for token in retained_tokens:
            input_embeddings.weight[new_vocab[token]].copy_(old_input[old_vocab[token]])
            if old_output is not None:
                output_embeddings.weight[new_vocab[token]].copy_(old_output[old_vocab[token]])
                if old_output_bias is not None:
                    output_embeddings.bias[new_vocab[token]].copy_(old_output_bias[old_vocab[token]])

        for token in new_tokens:
            target_id = new_vocab[token]
            ids = source_ids(token) if init_method == 'mean_of_constituents' else []

            if init_method == 'random':
                input_value = random_like(input_embeddings.weight[target_id], initializer_range)
                output_value = None if old_output is None else random_like(output_embeddings.weight[target_id], initializer_range)
            elif init_method == 'mean':
                input_value = input_mean
                output_value = output_mean
            else:
                input_value = input_mean if not ids else old_input[ids].float().mean(dim=0).to(old_input.dtype)
                output_value = (
                    None if old_output is None else
                    (output_mean if not ids else old_output[ids].float().mean(dim=0).to(old_output.dtype))
                )

            input_embeddings.weight[target_id].copy_(input_value)
            if output_value is not None:
                output_embeddings.weight[target_id].copy_(output_value)
                if output_embeddings.bias is not None:
                    output_embeddings.bias[target_id].zero_()

    if tied:
        model.tie_weights()

    moved = sum(old_vocab[token] != new_vocab[token] for token in retained_tokens)
    return {
        'init_method': init_method,
        'previous_rows': current_rows,
        'final_vocab_size': final_vocab_size,
        'final_rows': model.get_input_embeddings().weight.shape[0],
        'retained_tokens': len(retained_tokens),
        'moved_tokens': moved,
        'new_tokens': len(new_tokens),
        'removed_tokens': len(removed_tokens),
        'constituent_fallbacks': constituent_fallbacks,
    }

In [ ]:
def load_fresh_model():
    model_kwargs = {'torch_dtype': DTYPE, 'trust_remote_code': True}
    if torch.cuda.is_available():
        model_kwargs['device_map'] = 'auto'
    return AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_kwargs)


new_id_to_token = {token_id: token for token, token_id in diff['new_tokens'].items()}
observed_new_ids = sorted({
    token_id
    for text in CORPUS
    for token_id in target_tokenizer.encode(text, add_special_tokens=False)
    if token_id in new_id_to_token
})
assert observed_new_ids, 'No added token is reachable on the training corpus'
first_new_id = observed_new_ids[0]
first_new_token = new_id_to_token[first_new_id]
print('tracked reachable new token:', repr(first_new_token), first_new_id)
initialization_results = {}

for method in INITIALIZATION_METHODS:
    model = load_fresh_model()
    source_weights = model.get_input_embeddings().weight.detach().clone()
    unused_padding_start = len(target_tokenizer.get_vocab())
    unused_padding_before = source_weights[unused_padding_start:].clone()
    report = initialize_extended_token_embeddings(
        model=model,
        base_tokenizer=base_tokenizer,
        target_tokenizer=target_tokenizer,
        init_method=method,
    )

    vector = model.get_input_embeddings().weight[first_new_id].detach().float().cpu()
    old_vocab_ids = sorted(set(base_tokenizer.get_vocab().values()))
    if method == 'mean':
        expected = source_weights[old_vocab_ids].float().mean(dim=0).cpu()
    else:
        constituent_ids = [
            piece.id for piece in base_tokenizer._tokenizer.model.tokenize(first_new_token)
        ]
        assert constituent_ids
        expected = source_weights[constituent_ids].float().mean(dim=0).cpu()
    assert torch.allclose(vector, expected, atol=2e-3, rtol=2e-3)
    unused_padding_after = model.get_input_embeddings().weight[unused_padding_start:].detach()
    assert torch.equal(unused_padding_before, unused_padding_after)
    assert report['final_rows'] >= report['final_vocab_size']
    initialization_results[method] = {'report': report, 'vector': vector}
    print(method, report)

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

distance = torch.norm(
    initialization_results['mean']['vector'] -
    initialization_results['mean_of_constituents']['vector']
).item()
print(f'L2 distance for {first_new_token!r}: {distance:.6f}')
assert distance > 0, 'The two methods unexpectedly produced the same vector'

In [ ]:
def model_device(model):
    return next(model.parameters()).device


def make_batch(tokenizer, texts, device):
    batch = tokenizer(
        texts,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
    )
    batch = {name: value.to(device) for name, value in batch.items()}
    labels = batch['input_ids'].clone()
    labels[batch['attention_mask'] == 0] = -100
    batch['labels'] = labels
    return batch


def train_for_steps(model, tokenizer, steps=TRAIN_STEPS):
    model.train()
    trainable = [parameter for parameter in model.parameters() if parameter.requires_grad]
    if not trainable:
        raise RuntimeError('No trainable parameters')

    optimizer = torch.optim.AdamW(trainable, lr=LEARNING_RATE)
    device = model_device(model)
    losses = []

    for step in range(steps):
        texts = CORPUS[step * 2:(step + 1) * 2]
        batch = make_batch(tokenizer, texts, device)
        loss = model(**batch).loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        losses.append(float(loss.detach().cpu()))

    return losses


def freeze_all_except_embeddings(model):
    for parameter in model.parameters():
        parameter.requires_grad_(False)

    input_embeddings = model.get_input_embeddings()
    output_embeddings = model.get_output_embeddings()
    input_embeddings.weight.requires_grad_(True)

    if output_embeddings is not None:
        output_embeddings.weight.requires_grad_(True)
        if output_embeddings.bias is not None:
            output_embeddings.bias.requires_grad_(True)


def named_trainable_parameters(model):
    return [name for name, parameter in model.named_parameters() if parameter.requires_grad]


def find_default_lora_target(model):
    preferred_suffixes = ('q_proj', 'c_attn', 'query')
    names = [name for name, module in model.named_modules() if isinstance(module, torch.nn.Linear)]
    for suffix in preferred_suffixes:
        if any(name.endswith(suffix) for name in names):
            return suffix
    if not names:
        raise ValueError('Could not find a Linear module for the PEFT control adapter')
    return names[0].split('.')[-1]

## Full embedding-only training

This freezes every parameter except the input embeddings and output head. For tied embeddings, these refer to the same underlying weight.

In [ ]:
full_model = load_fresh_model()
full_report = initialize_extended_token_embeddings(
    model=full_model,
    base_tokenizer=base_tokenizer,
    target_tokenizer=target_tokenizer,
    init_method=TRAINING_METHOD,
)

freeze_all_except_embeddings(full_model)
frozen_name, frozen_parameter = next(
    (name, parameter) for name, parameter in full_model.named_parameters()
    if not parameter.requires_grad
)
frozen_before = frozen_parameter.detach().cpu().clone()
before_full = full_model.get_input_embeddings().weight[first_new_id].detach().float().cpu().clone()
print('full trainable parameters:', named_trainable_parameters(full_model)[:10])
full_losses = train_for_steps(full_model, target_tokenizer)
after_full = full_model.get_input_embeddings().weight[first_new_id].detach().float().cpu()

print('full report:', full_report)
print('full losses:', full_losses)
print('new-token row changed:', not torch.allclose(before_full, after_full))
assert not torch.allclose(before_full, after_full)
assert torch.equal(frozen_before, dict(full_model.named_parameters())[frozen_name].detach().cpu())

full_dir = WORK_DIR / 'embedding_only_full_model'
full_model.save_pretrained(full_dir, safe_serialization=True)
target_tokenizer.save_pretrained(full_dir)
trained_full_vector = after_full.clone()
del full_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

full_reload_kwargs = {'torch_dtype': DTYPE, 'trust_remote_code': True}
if torch.cuda.is_available():
    full_reload_kwargs['device_map'] = 'auto'
full_reloaded = AutoModelForCausalLM.from_pretrained(full_dir, **full_reload_kwargs)
reloaded_full_vector = full_reloaded.get_input_embeddings().weight[first_new_id].detach().float().cpu()
assert torch.equal(trained_full_vector, reloaded_full_vector)
reload_batch = make_batch(target_tokenizer, [CORPUS[0]], model_device(full_reloaded))
with torch.no_grad():
    full_reload_loss = full_reloaded(**reload_batch).loss
assert torch.isfinite(full_reload_loss)
print('full-model reload loss:', float(full_reload_loss.cpu()))
del full_reloaded
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## PEFT training with trainable embeddings

`modules_to_save` keeps the full input embedding and output-head modules trainable and serializes them with the adapter. The small LoRA target is included because PEFT's LoRA configuration requires at least one target module.

In [ ]:
peft_base_model = load_fresh_model()
peft_report = initialize_extended_token_embeddings(
    model=peft_base_model,
    base_tokenizer=base_tokenizer,
    target_tokenizer=target_tokenizer,
    init_method=TRAINING_METHOD,
)

input_name = resolve_module_name(peft_base_model, peft_base_model.get_input_embeddings())
output_module = peft_base_model.get_output_embeddings()
output_name = None if output_module is None else resolve_module_name(peft_base_model, output_module)
modules_to_save = list(dict.fromkeys([name for name in [input_name, output_name] if name]))
lora_target = find_default_lora_target(peft_base_model)

peft_config = LoraConfig(
    r=4,
    lora_alpha=8,
    lora_dropout=0.0,
    target_modules=[lora_target],
    modules_to_save=modules_to_save,
    task_type='CAUSAL_LM',
)
peft_model = get_peft_model(peft_base_model, peft_config)

trainable_before = named_trainable_parameters(peft_model)
print('embedding modules kept by PEFT:', modules_to_save)
print('LoRA control target:', lora_target)
print('sample trainable parameters:', trainable_before[:20])
assert any('modules_to_save' in name for name in trainable_before)
assert any('lora_' in name for name in trainable_before)

before_peft = peft_model.get_input_embeddings().weight[first_new_id].detach().float().cpu().clone()
peft_losses = train_for_steps(peft_model, target_tokenizer)
after_peft = peft_model.get_input_embeddings().weight[first_new_id].detach().float().cpu()

print('PEFT report:', peft_report)
print('PEFT losses:', peft_losses)
print('new-token row changed:', not torch.allclose(before_peft, after_peft))
assert not torch.allclose(before_peft, after_peft)
trained_peft_vector = after_peft.clone()

## Save, merge, reload, and verify

The adapter is saved with its trainable embedding modules. `merge_and_unload()` folds LoRA weights into the base model while preserving those saved embedding modules.

In [ ]:
adapter_dir = WORK_DIR / 'peft_adapter'
merged_dir = WORK_DIR / 'merged_model'

peft_model.save_pretrained(adapter_dir, safe_serialization=True)
target_tokenizer.save_pretrained(adapter_dir)

del peft_model
del peft_base_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

adapter_base = load_fresh_model()
initialize_extended_token_embeddings(
    model=adapter_base,
    base_tokenizer=base_tokenizer,
    target_tokenizer=target_tokenizer,
    init_method=TRAINING_METHOD,
)
reloaded_adapter = PeftModel.from_pretrained(adapter_base, adapter_dir)
adapter_vector = reloaded_adapter.get_input_embeddings().weight[first_new_id].detach().float().cpu()
assert torch.equal(trained_peft_vector, adapter_vector)
adapter_batch = make_batch(target_tokenizer, [CORPUS[0]], model_device(reloaded_adapter))
with torch.no_grad():
    adapter_loss = reloaded_adapter(**adapter_batch).loss
assert torch.isfinite(adapter_loss)
print('adapter reload loss:', float(adapter_loss.cpu()))

merged_model = reloaded_adapter.merge_and_unload()
merged_model.save_pretrained(merged_dir, safe_serialization=True)
target_tokenizer.save_pretrained(merged_dir)

del reloaded_adapter
del adapter_base
del merged_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

reloaded_tokenizer = AutoTokenizer.from_pretrained(merged_dir, trust_remote_code=True)
reloaded_model = AutoModelForCausalLM.from_pretrained(
    merged_dir,
    torch_dtype=DTYPE,
    trust_remote_code=True,
)

reloaded_rows = reloaded_model.get_input_embeddings().num_embeddings
reloaded_required_rows = required_embedding_rows(reloaded_tokenizer)
print({
    'adapter_dir': str(adapter_dir),
    'merged_dir': str(merged_dir),
    'reloaded_embedding_rows': reloaded_rows,
    'reloaded_required_rows': reloaded_required_rows,
})
assert reloaded_rows >= reloaded_required_rows
merged_vector = reloaded_model.get_input_embeddings().weight[first_new_id].detach().float().cpu()
assert torch.equal(trained_peft_vector, merged_vector)

prompt = 'भारत में भाषा मॉडल'
inputs = reloaded_tokenizer(prompt, return_tensors='pt')
inputs = {name: value.to(model_device(reloaded_model)) for name, value in inputs.items()}
with torch.no_grad():
    logits = reloaded_model(**inputs).logits
print('reload forward-pass logits shape:', tuple(logits.shape))
del reloaded_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Pruning and ID-remapping experiment

The real leaf-frequency pruner rebuilds contiguous IDs after removal. This verifies the count-based shrink and copies a moved token's embedding from its old ID to its new ID.

In [ ]:
PRUNE_COUNT = 8
pruned_tokenizer = copy.deepcopy(target_tokenizer)
pruner = LeafFrequencyPruner().train(pruned_tokenizer, CORPUS)
tokens_to_prune = pruner.get_tokens_to_prune(pruned_tokenizer, n=PRUNE_COUNT)
pruner.prune(pruned_tokenizer, n=PRUNE_COUNT)
pruned_diff = tokenizer_diff(target_tokenizer, pruned_tokenizer)
print('tokens pruned:', tokens_to_prune)
print('tokens moved:', len(pruned_diff['moved_tokens']))
assert len(pruned_diff['removed_tokens']) == PRUNE_COUNT
assert pruned_diff['moved_tokens']

pruned_model = load_fresh_model()
initialize_extended_token_embeddings(
    model=pruned_model,
    base_tokenizer=base_tokenizer,
    target_tokenizer=target_tokenizer,
    init_method=TRAINING_METHOD,
)
pre_prune_weights = pruned_model.get_input_embeddings().weight.detach().clone()
moved_token, (old_id, new_id) = next(iter(pruned_diff['moved_tokens'].items()))
prune_report = initialize_extended_token_embeddings(
    model=pruned_model,
    base_tokenizer=target_tokenizer,
    target_tokenizer=pruned_tokenizer,
    init_method=TRAINING_METHOD,
)
assert torch.equal(
    pre_prune_weights[old_id],
    pruned_model.get_input_embeddings().weight[new_id],
)
expected_rows = (
    len(target_tokenizer.get_vocab())
    + len(pruned_diff['new_tokens'])
    - len(pruned_diff['removed_tokens'])
)
assert expected_rows == len(pruned_tokenizer.get_vocab())
assert pruned_model.get_input_embeddings().num_embeddings == expected_rows
print('pruning report:', prune_report)
print('verified moved token:', repr(moved_token), old_id, '->', new_id)

pruned_dir = WORK_DIR / 'pruned_model'
pruned_model.save_pretrained(pruned_dir, safe_serialization=True)
pruned_tokenizer.save_pretrained(pruned_dir)
pruned_reload = AutoModelForCausalLM.from_pretrained(
    pruned_dir, torch_dtype=DTYPE, trust_remote_code=True
)
assert pruned_reload.get_input_embeddings().num_embeddings == expected_rows
pruned_inputs = pruned_tokenizer(CORPUS[0], return_tensors='pt')
with torch.no_grad():
    pruned_logits = pruned_reload(**pruned_inputs).logits
assert torch.isfinite(pruned_logits).all()
print('pruned reload logits shape:', tuple(pruned_logits.shape))